<a href="https://colab.research.google.com/github/ninjafox250/yolo-model-training/blob/main/train_yolo_modelsV2-4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!rm -r /content/runs/detect/

In [ ]:
!pip install ultralytics scikit-learn iterative-stratification


In [ ]:
# Unzip images to a custom data folder
!unzip -q /content/dataset.zip -d /content/dataset

In [ ]:
# =====================
# IMPORTS
# =====================
import shutil
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# =====================
# PATHS (COLAB)
# =====================
SRC_IMG_DIR = Path("/content/dataset/images")
SRC_LBL_DIR = Path("/content/dataset/labels")

BASE_OUT = Path("/content/data")

TRAIN_IMG_DIR = BASE_OUT / "train/images"
TRAIN_LBL_DIR = BASE_OUT / "train/labels"
VAL_IMG_DIR = BASE_OUT / "validation/images"
VAL_LBL_DIR = BASE_OUT / "validation/labels"
HOLDOUT_IMG_DIR = BASE_OUT / "holdout/images"
HOLDOUT_LBL_DIR = BASE_OUT / "holdout/labels"

IMG_EXTS = [".jpg", ".jpeg", ".png"]

# =====================
# CLEAN (optional but recommended in notebooks)
# =====================
shutil.rmtree(BASE_OUT, ignore_errors=True)

# =====================
# CREATE DIRS
# =====================
for d in [TRAIN_IMG_DIR, TRAIN_LBL_DIR, VAL_IMG_DIR, VAL_LBL_DIR, HOLDOUT_IMG_DIR, HOLDOUT_LBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =====================
# LOAD DATA (image-first)
# =====================
image_files = []
labels_per_image = []

for img_path in SRC_IMG_DIR.iterdir():
    if img_path.suffix.lower() not in IMG_EXTS:
        continue

    lbl_path = SRC_LBL_DIR / f"{img_path.stem}.txt"

    if not lbl_path.exists():
        continue

    classes = set()
    with open(lbl_path) as f:
        for line in f:
            if line.strip():
                classes.add(int(line.split()[0]))

    image_files.append(img_path)
    labels_per_image.append(list(classes))

print(f"Total usable dataset: {len(image_files)}")

# =====================
# ENCODE LABELS
# =====================
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels_per_image)

# =====================
# STEP 1: HOLDOUT (10%)
# =====================
holdout_split = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.10,
)

trainval_idx, holdout_idx = next(holdout_split.split(image_files, Y))

trainval_images = [image_files[i] for i in trainval_idx]
trainval_labels = [labels_per_image[i] for i in trainval_idx]

holdout_images = [image_files[i] for i in holdout_idx]

print(f"Holdout: {len(holdout_images)}")
print(f"Train/Val pool: {len(trainval_images)}")

# =====================
# STEP 2: TRAIN/VAL (80/20)
# =====================
mlb2 = MultiLabelBinarizer()
Y_trainval = mlb2.fit_transform(trainval_labels)

train_split = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
)

train_idx, val_idx = next(train_split.split(trainval_images, Y_trainval))

train_images = [trainval_images[i] for i in train_idx]
val_images = [trainval_images[i] for i in val_idx]

print(f"Train: {len(train_images)}")
print(f"Validation: {len(val_images)}")

# =====================
# COPY FUNCTIONS
# =====================
def copy_yolo_pairs(image_list, img_dst, lbl_dst):
    for img_path in image_list:
        lbl_path = SRC_LBL_DIR / f"{img_path.stem}.txt"

        shutil.copy2(img_path, img_dst / img_path.name)
        shutil.copy2(lbl_path, lbl_dst / lbl_path.name)

# =====================
# EXECUTE COPY
# =====================
copy_yolo_pairs(train_images, TRAIN_IMG_DIR, TRAIN_LBL_DIR)
copy_yolo_pairs(val_images, VAL_IMG_DIR, VAL_LBL_DIR)
copy_yolo_pairs(holdout_images, HOLDOUT_IMG_DIR, HOLDOUT_LBL_DIR)

print("✅ Done.")

In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'test': 'holdout/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/dataset/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspectiv

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=/content/data/holdout/images save=True

In [ ]:
from datetime import datetime
from ultralytics import YOLO
import json

model = YOLO("runs/detect/train/weights/best.pt")
metrics = model.val(
    data="data.yaml",
    split="test",
    plots=False,
    verbose=False
)

print(metrics.results_dict)
with open("/content/metrics.json", "w") as f:
    json.dump(metrics.results_dict, f, indent=2)

now = datetime.now()
modelname = now.strftime("%b").lower() + f"-{now.day}-{now.strftime('%y')}_{now.hour - 4}"

!mkdir -p /content/$modelname
from datetime import datetime
import os
import shutil
import subprocess

now = datetime.now()
modelname = now.strftime("%b").lower() + f"-{now.day}-{now.strftime('%y')}_{now.hour - 4}"

base_path = f"/content/{modelname}"
zip_path = f"/content/{modelname}.zip"

# ---------------------
# Create folder
# ---------------------
os.makedirs(base_path, exist_ok=True)

# ---------------------
# Copy files
# ---------------------
shutil.copy(
    "/content/runs/detect/train/weights/best.pt",
    f"{base_path}/{modelname}.pt"
)

shutil.copy(
    "/content/metrics.json",
    f"{base_path}/metrics.json"
)

shutil.copytree(
    "/content/runs/detect/predict",
    f"{base_path}/predict",
    dirs_exist_ok=True
)

shutil.copytree(
    "/content/data/holdout",
    f"{base_path}/holdout",
    dirs_exist_ok=True
)

# ---------------------
# ZIP properly (IMPORTANT FIX)
# ---------------------
subprocess.run(
    ["zip", "-r", zip_path, modelname],
    cwd="/content",
    check=True
)

# ---------------------
# Copy to Drive
# ---------------------
shutil.copy(zip_path, "/content/drive/MyDrive/")

In [ ]:
import os
os.kill(os.getpid(), 9)